In [ ]:
!pip -q install telethon pandas pyarrow tqdm

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 12.1 MB/s eta 0:00:00


In [ ]:
import os, re, time, asyncio, json, uuid
from math import ceil
from datetime import timezone
import pandas as pd
from tqdm.auto import tqdm
from datetime import datetime, timezone

from pathlib import Path


In [ ]:
from telethon.sync import TelegramClient
from telethon.errors import FloodWaitError, RPCError
from telethon.tl.functions.channels import GetFullChannelRequest
from telethon.tl.types import MessageMediaPhoto, DocumentAttributeVideo, MessageMediaPoll

In [ ]:
API_ID = 25762635
API_HASH = '9461780508f89e5b4baccda5f38b6f03'
SESSION = "tg_collect_session"


In [ ]:
CHANNELS = [
    "moscowtoplive","ru2ch","mash","bazabazon","litvintm","plsbetenderly","instasamkacore","topor","Lepragram",
    "sms_future","whackdoor","bugfeature","aiaiai","bezposhady","Crypto_Woolf","br_dev","kristina_egiazarova14",
    "veronikastepanova20011","klientvsprav","FamilyBots","trendi","etoznakmag","Coin_Post","bankoffo","commerc","pekagame",
]


In [ ]:
TARGET_TOTAL = 11000
CHUNK_SIZE   = 500                 # как часто сбрасывать на диск
OUT_CSV      = "/content/tg_posts.csv"
OUT_PQ_DIR   = "/content/tg_posts_parquet"   # пишем part-файлами
SEEN_KEYS    = "/content/tg_posts_seen.csv"  # хранилище ключей для возобновления

In [ ]:
os.makedirs(OUT_PQ_DIR, exist_ok=True)

# --- загрузка уже сохранённых ключей (для резюма) ---
if os.path.exists(SEEN_KEYS):
    seen_df = pd.read_csv(SEEN_KEYS, dtype=str)
    seen = set(map(tuple, seen_df[["platform","channel_id","message_id"]].values.tolist()))
else:
    seen = set()

# --- служебные ---
url_regex = re.compile(r"https?://\S+")

def iso_utc(dt):
    if dt is None: return None
    if dt.tzinfo is None: dt = dt.replace(tzinfo=timezone.utc)
    return dt.astimezone(timezone.utc).isoformat()

def has_video(msg):
    if not msg or not msg.media: return False
    if getattr(msg, "video", None): return True
    doc = getattr(msg.media, "document", None)
    if doc and getattr(doc, "attributes", None):
        return any(isinstance(a, DocumentAttributeVideo) for a in doc.attributes)
    return False

def has_photo(msg):
    return isinstance(getattr(msg, "media", None), MessageMediaPhoto)

def has_link(msg):
    text = msg.message or ""
    if url_regex.search(text): return True
    return bool(getattr(msg, "entities", None))

def has_poll(msg):
    return isinstance(getattr(msg, "media", None), MessageMediaPoll)

def reactions_sum(msg):
    r = getattr(msg, "reactions", None)
    if not r: return None
    if getattr(r, "results", None):
        return sum(getattr(t, "count", 0) for t in r.results)
    if getattr(r, "totals", None):
        return sum(getattr(t, "count", 0) for t in r.totals)
    return None

def forward_source(msg):
    fwd = getattr(msg, "fwd_from", None)
    if not fwd: return None, None, None
    src_id = getattr(getattr(fwd, "from_id", None), "channel_id", None) or getattr(getattr(fwd, "from_id", None), "user_id", None)
    src_name = getattr(fwd, "from_name", None)
    src_msg_id = getattr(fwd, "channel_post", None)
    return src_id, src_name, src_msg_id

async def participants_count(client, entity):
    try:
        full = await client(GetFullChannelRequest(entity))
        return getattr(full.full_chat, "participants_count", None)
    except RPCError:
        return None

async def safe_iter_messages(client, entity, limit):
    fetched = 0
    while fetched < limit:
        remain = limit - fetched
        try:
            batch = []
            async for m in client.iter_messages(entity, limit=remain):
                batch.append(m)
                if len(batch) >= remain: break
            if not batch: break
            for m in batch: yield m
            fetched += len(batch)
        except FloodWaitError as e:
            await asyncio.sleep(e.seconds + 1)
        except RPCError:
            break

# --- инкрементальное сохранение ---
def coerce_types(df):
    num_cols = ["likes","comments","reposts","followers"]
    for c in num_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def save_chunk(chunk_records):
    if not chunk_records: return 0
    df = pd.DataFrame.from_records(chunk_records)
    if df.empty: return 0

    # фильтрация уже виденных ключей (межзапусковая дедупликация)
    df_keys = list(map(tuple, df[["platform","channel_id","message_id"]].astype(str).values.tolist()))
    mask_new = [k not in seen for k in df_keys]
    if not any(mask_new): return 0
    df = df[mask_new]
    if df.empty: return 0

    # внутричанковая дедупликация
    df.drop_duplicates(subset=["platform","channel_id","message_id"], inplace=True)

    df = coerce_types(df)

    # CSV append
    write_header = not os.path.exists(OUT_CSV)
    df.to_csv(OUT_CSV, mode="a", index=False, encoding="utf-8-sig", header=write_header)

    # Parquet part-файл
    part_name = f"part-{uuid.uuid4().hex}.parquet"
    df.to_parquet(os.path.join(OUT_PQ_DIR, part_name), index=False)

    # обновляем seen и файл ключей
    new_keys_df = df[["platform","channel_id","message_id"]].astype(str)
    new_keys_df.to_csv(SEEN_KEYS, mode="a", index=False, header=not os.path.exists(SEEN_KEYS))
    for k in map(tuple, new_keys_df.values.tolist()):
        seen.add(k)

    return len(df)


In [ ]:
# --- основной сбор с периодическим сбросом ---
async def main():
    buffer = []
    saved_total = 0
    per_channel = ceil(TARGET_TOTAL / max(1, len(CHANNELS)))

    async with TelegramClient(SESSION, API_ID, API_HASH) as client:
        for ch in tqdm(CHANNELS, desc="Channels"):
            try:
                entity = await client.get_entity(ch)
            except RPCError:
                continue

            subs = await participants_count(client, entity)
            username = getattr(entity, "username", None)
            chan_id = getattr(entity, "id", None)
            chan_title = getattr(entity, "title", None)

            cnt = 0
            async for msg in safe_iter_messages(client, entity, limit=per_channel):
                if msg is None or msg.id is None:
                    continue

                fwd_id, fwd_name, fwd_msg_id = forward_source(msg)
                likes    = reactions_sum(msg)
                comments = getattr(getattr(msg, "replies", None), "replies", None)
                reposts  = getattr(msg, "forwards", None)

                rec = {
                    "platform": "tg",
                    "source_name": chan_title,
                    "source_link": f"https://t.me/{username}" if username else None,
                    "followers": subs,
                    "channel_id": chan_id,
                    "channel_username": username,
                    "message_id": msg.id,
                    "date": iso_utc(msg.date),
                    "initial_date": iso_utc(msg.date),
                    "text": msg.message or "",
                    "text_len": len(msg.message or ""),
                    "likes": likes,
                    "comments": comments,
                    "reposts": reposts,
                    "has_photo": bool(has_photo(msg)),
                    "has_video": bool(has_video(msg)),
                    "has_link": bool(has_link(msg)),
                    "has_poll": bool(has_poll(msg)),
                    "is_ad": None,
                    "image_url": None,
                    "message_link": f"https://t.me/{username}/{msg.id}" if username else None,
                    "is_forwarded": bool(getattr(msg, "fwd_from", None) is not None),
                    "fwd_src_id": fwd_id,
                    "fwd_src_name": fwd_name,
                    "fwd_src_msg_id": fwd_msg_id,
                }
                buffer.append(rec)
                cnt += 1

                if len(buffer) >= CHUNK_SIZE:
                    saved = save_chunk(buffer)
                    buffer.clear()
                    saved_total += saved

                if cnt % 500 == 0:
                    await asyncio.sleep(1)

    # финальный сброс
    saved = save_chunk(buffer)
    saved_total += saved
    print(f"Saved incrementally: {saved_total} rows")
    print(f"CSV: {OUT_CSV}")
    print(f"Parquet dir: {OUT_PQ_DIR}")
    # при необходимости можно собрать head:
    if os.path.exists(OUT_CSV):
        df_head = pd.read_csv(OUT_CSV, nrows=3)
        display(df_head)

await main()

Please enter your phone (or bot token): +79250753893
Please enter the code you received: 26408
Please enter your password: ··········
Signed in successfully as Daria; remember to not break the ToS or you will risk an account ban!


Channels:   0%|          | 0/26 [00:00<?, ?it/s]

Saved incrementally: 11024 rows
CSV: /content/tg_posts.csv
Parquet dir: /content/tg_posts_parquet


,platform,source_name,source_link,followers,channel_id,channel_username,message_id,date,initial_date,text,...,has_video,has_link,has_poll,is_ad,image_url,message_link,is_forwarded,fwd_src_id,fwd_src_name,fwd_src_msg_id
0,tg,Москва с огоньком Live,https://t.me/moscowtoplive,101489,1699803165,moscowtoplive,32607,2025-08-14T09:02:35+00:00,2025-08-14T09:02:35+00:00,Муром: 150 миллионов рублей на подсветку моста...,...,True,True,False,NaN,NaN,https://t.me/moscowtoplive/32607,False,NaN,NaN,NaN
1,tg,Москва с огоньком Live,https://t.me/moscowtoplive,101489,1699803165,moscowtoplive,32606,2025-08-14T08:41:34+00:00,2025-08-14T08:41:34+00:00,Туристка из РФ погибла в результате ДТП в Егип...,...,False,True,False,NaN,NaN,https://t.me/moscowtoplive/32606,False,NaN,NaN,NaN
2,tg,Москва с огоньком Live,https://t.me/moscowtoplive,101489,1699803165,moscowtoplive,32605,2025-08-14T07:55:53+00:00,2025-08-14T07:55:53+00:00,Слова «миллион» и «миллиард» теперь можно прои...,...,False,True,False,NaN,NaN,https://t.me/moscowtoplive/32605,False,NaN,NaN,NaN


In [ ]:
#  dynamics

In [ ]:
INPUT_CSV   = "/content/tg_posts.csv"
DATE_TAG    = datetime.utcnow().date().isoformat()   #
OUTPUT_CSV  = "/content/tg_posts2.csv"                # обновляем поверх
OUTPUT_PQ   = "/content/tg_posts2.parquet"
BATCH_IDS   = 200

# === Helpers ===
TME_RE = re.compile(r"(?:https?://)?t\.me/(?:c/)?([A-Za-z0-9_]+)/?")

def extract_username(row):
    # 1) явный username
    u = str(row.get("channel_username", "")).strip()
    if u and u != "None" and u.lower() != "nan":
        return u.lstrip("@")
    # 2) из source_link
    s = str(row.get("source_link", "")).strip()
    m = TME_RE.search(s)
    if m:
        return m.group(1)
    # 3) из message_link
    ml = str(row.get("message_link", "")).strip()
    m2 = TME_RE.search(ml)
    if m2:
        return m2.group(1)
    return None

def reactions_sum(msg):
    r = getattr(msg, "reactions", None)
    if not r: return None
    if getattr(r, "results", None):
        return sum(getattr(t, "count", 0) for t in r.results)
    if getattr(r, "totals", None):
        return sum(getattr(t, "count", 0) for t in r.totals)
    return None

async def fetch_and_update():
    base = pd.read_csv(INPUT_CSV)
    if "message_id" not in base.columns:
        raise ValueError("В INPUT_CSV нет столбца 'message_id'")

    # канонический username
    base["canon_username"] = base.apply(extract_username, axis=1)
    base = base[base["canon_username"].notna()].copy()

    # новые колонки на дату
    col_l = f"likes_{DATE_TAG}"
    col_c = f"comments_{DATE_TAG}"
    col_r = f"reposts_{DATE_TAG}"
    col_t = f"collected_at_{DATE_TAG}"
    for col in (col_l, col_c, col_r, col_t):
        if col not in base.columns:
            base[col] = pd.Series([pd.NA]*len(base))

    # группировка: username -> список message_id
    groups = (
        base[["canon_username","message_id"]]
        .dropna()
        .astype({"message_id":"int64"})
        .groupby("canon_username")["message_id"]
        .apply(list)
        .to_dict()
    )

    async with TelegramClient(SESSION, API_ID, API_HASH) as client:
        for uname, mids in tqdm(groups.items(), desc="Channels"):
            # стараемся резолвить по ссылке (надёжнее)
            try:
                entity = await client.get_entity(f"https://t.me/{uname}")
            except (RPCError, ValueError):
                continue

            # батчами по ids
            for i in range(0, len(mids), BATCH_IDS):
                batch = mids[i:i+BATCH_IDS]
                # повтор при FloodWait
                while True:
                    try:
                        msgs = await client.get_messages(entity, ids=batch)
                        break
                    except FloodWaitError as e:
                        await asyncio.sleep(e.seconds + 1)
                    except RPCError:
                        msgs = []
                        break

                now_iso = datetime.now(timezone.utc).isoformat()

                for msg in msgs:
                    if msg is None:
                        continue
                    # точная маска: username + message_id
                    mask = (base["canon_username"] == uname) & (base["message_id"].astype(int) == int(msg.id))

                    # новые значения
                    val_l = reactions_sum(msg)
                    val_c = getattr(getattr(msg, "replies", None), "replies", None)
                    val_r = getattr(msg, "forwards", None)

                    # дозаполнение только NaN
                    base.loc[mask & base[col_l].isna(), col_l] = val_l
                    base.loc[mask & base[col_c].isna(), col_c] = val_c
                    base.loc[mask & base[col_r].isna(), col_r] = val_r
                    base.loc[mask & base[col_t].isna(), col_t] = now_iso

    # типы
    for c in [col_l, col_c, col_r, "likes", "comments", "reposts", "followers"]:
        if c in base.columns:
            base[c] = pd.to_numeric(base[c], errors="coerce")

    # запись
    base.drop(columns=["canon_username"], inplace=True)
    base.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    base.to_parquet(OUTPUT_PQ, index=False)
    print(f"Updated: {OUTPUT_CSV}")
    print(f"Parquet: {OUTPUT_PQ}")
    print("New columns:", col_l, col_c, col_r, col_t)

await fetch_and_update()

Please enter your phone (or bot token): +79250753893
Please enter the code you received: 15775
Please enter your password: ··········
Signed in successfully as Daria; remember to not break the ToS or you will risk an account ban!


Channels:   0%|          | 0/21 [00:00<?, ?it/s]

Updated: /content/tg_posts2.csv
Parquet: /content/tg_posts2.parquet
New columns: likes_2025-08-18 comments_2025-08-18 reposts_2025-08-18 collected_at_2025-08-18


In [ ]:
INPUT_CSV   = "/content/tg_pairs.csv"              # файл с колонками из примера
OUTPUT_CSV  = "/content/tg_pairs.csv"              # обновляем поверх
OUTPUT_PQ   = "/content/tg_pairs.parquet"

# Дата снапшота для имён колонок. Оставь пустым — будет UTC сегодня.
SNAPSHOT_DATE = ""  # например "2025-08-15"
DATE_TAG = SNAPSHOT_DATE or datetime.utcnow().date().isoformat()

BATCH_IDS = 200

# === Helpers ===
TME_RE = re.compile(r"(?:https?://)?t\.me/(?:c/)?([A-Za-z0-9_]+)/(\d+)")

def parse_link(link: str):
    if not isinstance(link, str):
        return None, None
    m = TME_RE.search(link.strip())
    if not m:
        return None, None
    return m.group(1), int(m.group(2))

def to_int(x):
    try:
        # допускаем float вида 32041.0
        return int(float(x))
    except Exception:
        return None

def reactions_sum(msg):
    r = getattr(msg, "reactions", None)
    if not r: return None
    if getattr(r, "results", None):
        return sum(getattr(t, "count", 0) for t in r.results)
    if getattr(r, "totals", None):
        return sum(getattr(t, "count", 0) for t in r.totals)
    return None

async def fetch_messages(client, uname, ids):
    out = {}
    for i in range(0, len(ids), BATCH_IDS):
        batch = ids[i:i+BATCH_IDS]
        while True:
            try:
                entity = await client.get_entity(f"https://t.me/{uname}")
                msgs = await client.get_messages(entity, ids=batch)
                break
            except FloodWaitError as e:
                await asyncio.sleep(e.seconds + 1)
            except (RPCError, ValueError):
                msgs = []
                break
        for msg in msgs:
            if msg is None:
                continue
            out[int(msg.id)] = msg
    return out

async def update_originals():
    df = pd.read_csv(INPUT_CSV)

    # Проверка обязательных колонок
    need_cols = ["original_channel","original_post_id","original_post_link"]
    for c in need_cols:
        if c not in df.columns:
            df[c] = pd.NA

    # Целевые колонки на дату
    col_l = f"original_likes_{DATE_TAG}"
    col_c = f"original_comments_{DATE_TAG}"
    col_r = f"original_reposts_{DATE_TAG}"
    col_t = f"original_collected_at_{DATE_TAG}"
    for col in (col_l, col_c, col_r, col_t):
        if col not in df.columns:
            df[col] = pd.Series([pd.NA]*len(df))

    # Извлекаем username+id: приоритет link, иначе channel+post_id
    rows = []
    for idx, row in df.iterrows():
        uname, mid = parse_link(row.get("original_post_link", ""))
        if uname and mid:
            rows.append((idx, uname, mid))
            continue
        uname2 = str(row.get("original_channel", "")).strip().lstrip("@")
        mid2 = to_int(row.get("original_post_id", None))
        if uname2 and mid2:
            rows.append((idx, uname2, mid2))

    # Группировка по username
    pool = {}
    for idx, uname, mid in rows:
        pool.setdefault(uname, {"idx": [], "ids": []})
        pool[uname]["idx"].append(idx)
        pool[uname]["ids"].append(mid)

    async with TelegramClient(SESSION, API_ID, API_HASH) as client:
        now_iso = datetime.now(timezone.utc).isoformat()

        for uname, bundle in tqdm(pool.items(), desc="Channels"):
            ids = sorted(set(bundle["ids"]))
            id_to_msg = await fetch_messages(client, uname, ids)

            # Обновляем только пустые значения
            for idx, mid in zip(bundle["idx"], bundle["ids"]):
                msg = id_to_msg.get(int(mid))
                if not msg:
                    continue
                if pd.isna(df.at[idx, col_l]): df.at[idx, col_l] = reactions_sum(msg)
                if pd.isna(df.at[idx, col_c]): df.at[idx, col_c] = getattr(getattr(msg, "replies", None), "replies", None)
                if pd.isna(df.at[idx, col_r]): df.at[idx, col_r] = getattr(msg, "forwards", None)
                if pd.isna(df.at[idx, col_t]): df.at[idx, col_t] = now_iso

    # Приведение типов числовых новых колонок
    for c in (col_l, col_c, col_r):
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # Сохранение
    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    df.to_parquet(OUTPUT_PQ, index=False)
    print("Updated:", OUTPUT_CSV)
    print("Parquet:", OUTPUT_PQ)
    print("Added:", col_l, col_c, col_r, col_t)

await update_originals()

Channels:   0%|          | 0/23 [00:00<?, ?it/s]

Updated: /content/tg_pairs.csv
Parquet: /content/tg_pairs.parquet
Added: original_likes_2025-08-15 original_comments_2025-08-15 original_reposts_2025-08-15 original_collected_at_2025-08-15


In [ ]:

# --- пути ---
POSTS_CSV = "/content/tg_posts.csv"      # твой общий постовый датасет TG
PAIRS_CSV = "/content/tg_pairs.csv"      # файл с original_* и forward_*
OUT_CSV   = "/content/tg_merged.csv"
OUT_PQ    = "/content/tg_merged.parquet"

# --- утилиты ---
TME_RE = re.compile(r"(?:https?://)?t\.me/(?:c/)?([A-Za-z0-9_]+)/(\d+)")

def parse_tme(link: str):
    if not isinstance(link, str):
        return None, None
    m = TME_RE.search(link.strip())
    if not m:
        return None, None
    return m.group(1), int(m.group(2))

def norm_username(u):
    if u is None:
        return None
    s = str(u).strip()
    if not s or s.lower() in ("none", "nan"):
        return None
    return s.lstrip("@")

def safe_int(x):
    try:
        return int(float(x))
    except Exception:
        return None

# --- загрузка ---
posts = pd.read_csv(POSTS_CSV)
pairs = pd.read_csv(PAIRS_CSV)

# --- ключи в posts: username + message_id ---
# 1) username из channel_username или из source_link/message_link
def extract_username_posts(row):
    u = norm_username(row.get("channel_username"))
    if u:
        return u
    for col in ("source_link", "message_link"):
        u2, _ = parse_tme(row.get(col, ""))
        if u2:
            return u2
    return None

posts["__uname__"] = posts.apply(extract_username_posts, axis=1)
posts["__mid__"]   = posts["message_id"].apply(safe_int) if "message_id" in posts.columns else None

# отфильтруем, где ключ собрать удалось
posts_keyed = posts[posts["__uname__"].notna() & posts["__mid__"].notna()].copy()

# --- ключи в pairs: берём из original_post_link, иначе из original_channel + original_post_id ---
def extract_uname_mid_pairs(row):
    u, m = parse_tme(row.get("original_post_link", ""))
    if u and m:
        return u, m
    u2 = norm_username(row.get("original_channel"))
    m2 = safe_int(row.get("original_post_id"))
    return (u2, m2)

pairs["__uname__"], pairs["__mid__"] = zip(*pairs.apply(extract_uname_mid_pairs, axis=1))
pairs_keyed = pairs[pairs["__uname__"].notna() & pairs["__mid__"].notna()].copy()

# --- выделим столбцы из pairs, которые нужно добавить в конец ---
# берём всё, что относится к форвардам и метрикам сопоставления
forward_cols = [c for c in pairs_keyed.columns if c.startswith("forward_")]
meta_cols = [c for c in [
    "reason","similarity","viewsCount","sharesCount","commentsCount",
    "reactionsCount","forwardsCount","mentionsCount"
] if c in pairs_keyed.columns]

# также захватим любые уже рассчитанные original_* снапшоты (на дату), если есть
original_snapshot_cols = [c for c in pairs_keyed.columns if c.startswith("original_")]

cols_to_add = original_snapshot_cols + forward_cols + meta_cols

pairs_slim = pairs_keyed[["__uname__","__mid__"] + cols_to_add].copy()

# если в pairs дубли по ключу — оставим первый (или агрегируй по необходимости)
pairs_slim = pairs_slim.drop_duplicates(subset=["__uname__","__mid__"], keep="first")

# --- merge (left) ---
merged = posts_keyed.merge(pairs_slim, on=["__uname__","__mid__"], how="left", suffixes=("",""))

# признак наличия пары
merged["has_forward_pair"] = merged[forward_cols].notna().any(axis=1) if forward_cols else False

# --- порядок столбцов: исходные posts -> затем добавляемые из pairs ---
base_cols = [c for c in posts.columns]  # в исходном порядке
add_cols  = [c for c in original_snapshot_cols + forward_cols + meta_cols if c in merged.columns]
final_cols = base_cols + add_cols + (["has_forward_pair"] if "has_forward_pair" in merged.columns else [])

# могут быть технические ключи в base_cols; уберём их из финального списка и добавим в конец для отладки
tech_cols = ["__uname__","__mid__"]
final_cols = [c for c in final_cols if c not in tech_cols] + [c for c in tech_cols if c in merged.columns]

merged = merged[final_cols]

# --- сохранение ---
merged.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
merged.to_parquet(OUT_PQ, index=False)

print(f"Saved: {OUT_CSV}")
print(f"Saved: {OUT_PQ}")
print("Rows:", len(merged))

posts total: 8904
posts with normalized link: 8904
posts with uname+mid: 8904
pairs total: 1790
pairs with normalized link: 1790
pairs with uname+mid: 1790


ValueError: The column label '__olink__' is not unique.

In [ ]:
# ====== Colab setup ======
!pip -q install pandas pyarrow python-dateutil

import re
import pandas as pd
import numpy as np
from datetime import datetime, timezone
from dateutil import parser
from pathlib import Path

# ====== Пути ======
TG_CSV = "/content/tg_merged.csv"      # Telegram
VK_CSV = "/content/vk_posts.csv"       # VK
OUT_LONG_CSV = "/content/social_long.csv"
OUT_LONG_PQ  = "/content/social_long.parquet"

# Если нужен топ по ER на конкретную дату — укажи дату (YYYY-MM-DD). Иначе оставь None.
DATE_FILTER = None     # например "2025-08-15"
TOP_K = 100
TOP_DIR = Path("/content/er_outputs"); TOP_DIR.mkdir(parents=True, exist_ok=True)

# ====== Утилиты ======
METRIC_PATTERNS = [
    r'^(likes|comments|reposts)_(\d{4}-\d{2}-\d{2})$',
    r'^original_(likes|comments|reposts)_(\d{4}-\d{2}-\d{2})$'
]
COMPILED = re.compile("|".join(f"(?:{p})" for p in METRIC_PATTERNS))

def epoch_to_iso(x):
    try:
        # VK может давать epoch; TG уже ISO → парсим в обоих случаях
        if str(x).strip().isdigit():
            x = int(x)
            return datetime.fromtimestamp(x, tz=timezone.utc).isoformat()
        return parser.isoparse(str(x)).astimezone(timezone.utc).isoformat()
    except Exception:
        return pd.NA

def norm_bool(x):
    if pd.isna(x): return pd.NA
    if isinstance(x, str):
        s = x.strip().lower()
        if s in ("true","1","yes","y","t"):  return True
        if s in ("false","0","no","n","f"):  return False
    try:
        return bool(x)
    except Exception:
        return pd.NA

def extract_link_tg(row):
    ml = row.get("message_link")
    if pd.notna(ml) and str(ml).startswith("http"):
        return ml
    return row.get("source_link")

def build_link_vk(row):
    # оставляем ссылку на источник; прямую ссылку на пост VK можно не строить
    return row.get("source_link")

def collect_metric_cols(cols):
    out = []
    for c in cols:
        m = COMPILED.match(c)
        if not m:
            continue
        # вытаскиваем две последние ненулевые группы: metric, date
        groups = [g for g in m.groups() if g is not None]
        if len(groups) == 2:
            out.append((c, groups[0], groups[1]))  # (orig_col, metric, date)
    return out

def melt_metrics(df, id_cols, metric_cols_spec):
    if not metric_cols_spec:
        return pd.DataFrame(columns=id_cols + ["metric","metric_date","value"])
    keep = id_cols + [c for (c,_,_) in metric_cols_spec]
    # защита от дубликатов колонок
    df = df.loc[:, ~pd.Index(df.columns).duplicated()]
    wide = df[keep].copy()
    long = wide.melt(id_vars=id_cols, var_name="orig_col", value_name="value")
    meta = pd.DataFrame(metric_cols_spec, columns=["orig_col","metric","metric_date"])
    long = long.merge(meta, on="orig_col", how="left").drop(columns=["orig_col"])
    long["metric_date"] = pd.to_datetime(long["metric_date"], errors="coerce").dt.date.astype("string")
    long["value"] = pd.to_numeric(long["value"], errors="coerce")
    return long

# ====== Telegram ======
tg = pd.read_csv(TG_CSV)
tg_base = pd.DataFrame({
    "platform":    "tg",
    "post_id":     tg.get("message_id"),
    "published_at": (tg["initial_date"] if "initial_date" in tg.columns else tg["date"]).apply(epoch_to_iso),
    "source_name": tg.get("source_name"),
    "source_link": tg.get("source_link"),
    "link":        tg.apply(extract_link_tg, axis=1),
    "followers":   pd.to_numeric(tg.get("followers"), errors="coerce"),
    "has_photo":   tg.get("has_photo").apply(norm_bool) if "has_photo" in tg.columns else pd.NA,
    "has_video":   tg.get("has_video").apply(norm_bool) if "has_video" in tg.columns else pd.NA,
    "has_link":    tg.get("has_link").apply(norm_bool)  if "has_link"  in tg.columns else pd.NA,
    "is_ad":       tg.get("is_ad"),
    "text":        tg.get("text")
})
tg_full = pd.concat([tg_base, tg], axis=1)
tg_full = tg_full.loc[:, ~tg_full.columns.duplicated()]
tg_specs = collect_metric_cols(tg_full.columns)
tg_long = melt_metrics(tg_full, id_cols=list(tg_base.columns.unique()), metric_cols_spec=tg_specs)

# ====== VK ======
vk = pd.read_csv(VK_CSV)
vk_base = pd.DataFrame({
    "platform":    "vk",
    "post_id":     vk.get("post_id"),
    "published_at": (vk["initial_date"] if "initial_date" in vk.columns else vk["date"]).apply(epoch_to_iso),
    "source_name": vk.get("source_name"),
    "source_link": vk.get("source_link"),
    "link":        vk.apply(build_link_vk, axis=1),
    "followers":   pd.to_numeric(vk.get("followers"), errors="coerce"),
    "has_photo":   vk.get("has_photo").apply(norm_bool) if "has_photo" in vk.columns else pd.NA,
    "has_video":   vk.get("has_video").apply(norm_bool) if "has_video" in vk.columns else pd.NA,
    "has_link":    vk.get("has_link").apply(norm_bool)  if "has_link"  in vk.columns else pd.NA,
    "is_ad":       vk.get("is_ad"),
    "text":        vk.get("text")
})
vk_full = pd.concat([vk_base, vk], axis=1)
vk_full = vk_full.loc[:, ~vk_full.columns.duplicated()]
vk_specs = collect_metric_cols(vk_full.columns)
vk_long = melt_metrics(vk_full, id_cols=list(vk_base.columns.unique()), metric_cols_spec=vk_specs)

# ====== Объединение и ER ======
long_all = pd.concat([tg_long, vk_long], ignore_index=True)

# упорядочим столбцы
order = [
    "platform","post_id","metric","metric_date","value",
    "published_at","source_name","source_link","link",
    "followers","has_photo","has_video","has_link","is_ad","text"
]
long_all = long_all[order]

# engagement rate
long_all["engagement_rate"] = np.where(
    (pd.to_numeric(long_all["followers"], errors="coerce") > 0),
    long_all["value"] / pd.to_numeric(long_all["followers"], errors="coerce"),
    np.nan
)

# ====== Сохранение long ======
long_all.to_csv(OUT_LONG_CSV, index=False, encoding="utf-8-sig")
long_all.to_parquet(OUT_LONG_PQ, index=False)

print("Saved:", OUT_LONG_CSV)
print("Saved:", OUT_LONG_PQ)
print("Rows:", len(long_all))

# ====== Топы по ER на дату (опционально) ======
if DATE_FILTER:
    cut = long_all[long_all["metric_date"] == DATE_FILTER].copy()
    cut.drop_duplicates(subset=["platform","post_id","metric","metric_date"], inplace=True)

    def dump_top(df, name):
        cols = [
            "platform","metric","metric_date","engagement_rate","value",
            "followers","post_id","published_at","source_name","source_link","link",
            "has_photo","has_video","has_link","is_ad","text"
        ]
        out = df[cols].sort_values(by=["engagement_rate","value"], ascending=[False, False]).head(TOP_K)
        out.to_csv(TOP_DIR / f"{name}.csv", index=False, encoding="utf-8-sig")
        return out

    top_overall = dump_top(cut, f"top{TOP_K}_overall_{DATE_FILTER}")
    top_tg      = dump_top(cut[cut["platform"]=="tg"], f"top{TOP_K}_tg_{DATE_FILTER}")
    top_vk      = dump_top(cut[cut["platform"]=="vk"], f"top{TOP_K}_vk_{DATE_FILTER}")

    print(f"Tops saved to: {str(TOP_DIR)}")



Saved: /content/social_long.csv
Saved: /content/social_long.parquet
Rows: 90795


In [ ]:
# WIDE из TG+VK с сохранением исходных полей и ER по репостам.
import os, re, numpy as np, pandas as pd

# --- пути (проверь имена файлов) ---
TG_CSV = "/content/tg_merged.csv"
VK_CSV = "/content/vk_posts.csv"   # твой VK CSV
OUT_CSV = "/content/social_wide.csv"
OUT_PQ  = "/content/social_wide.parquet"

# --- регэксп для срезов ---
SNAP_PAT = re.compile(r"^(?:original_)?(likes|comments|reposts)_(\d{4}-\d{2}-\d{2})$")
COLLECTED_PAT = re.compile(r"^(?:original_)?collected_at_(\d{4}-\d{2}-\d{2})$")

def read_df(path):
    if not os.path.exists(path): return pd.DataFrame()
    return pd.read_parquet(path) if path.endswith(".parquet") else pd.read_csv(path)

# исходные поля, которые хотим сохранить как есть
VK_KEEP = [
    "group_id","post_id","from_id","text","date","initial_date","likes",
    "comments","reposts","image_url","followers","source_name","source_link",
    "is_ad","has_photo","has_video","has_link","has_poll"
]
TG_KEEP = [
    "platform","source_name","source_link","followers","channel_id","channel_username",
    "message_id","date","initial_date","text","text_len","likes","comments","reposts",
    "has_photo","has_video","has_link","has_poll","is_ad","image_url","message_link",
    "is_forwarded","fwd_src_id","fwd_src_name","fwd_src_msg_id"
]

def ensure_cols(df, cols):
    for c in cols:
        if c not in df.columns:
            df[c] = pd.NA
    return df

def base_block_vk(vk):
    vk = ensure_cols(vk, VK_KEEP)
    out = pd.DataFrame({
        # базовые
        "platform":     "vk",
        "source_name":  vk["source_name"],
        "source_link":  vk["source_link"],
        "followers":    pd.to_numeric(vk["followers"], errors="coerce"),
        "post_id":      vk["post_id"],
        "published_at": vk["initial_date"].fillna(vk["date"]),
        "text":         vk["text"],
        "has_photo":    vk["has_photo"],
        "has_video":    vk["has_video"],
        "has_link":     vk["has_link"],
        "has_poll":     vk["has_poll"],
        "is_ad":        vk["is_ad"],
        "link":         vk["source_link"],
        # оставляем оригинальные поля VK
        "vk_group_id":  vk["group_id"],
        "vk_from_id":   vk["from_id"],
        "image_url":    vk["image_url"],
    })
    return out

def base_block_tg(tg):
    tg = ensure_cols(tg, TG_KEEP)
    out = pd.DataFrame({
        "platform":     "tg",
        "source_name":  tg["source_name"],
        "source_link":  tg["source_link"],
        "followers":    pd.to_numeric(tg["followers"], errors="coerce"),
        "post_id":      tg["message_id"],
        "published_at": tg["initial_date"].fillna(tg["date"]),
        "text":         tg["text"],
        "has_photo":    tg["has_photo"],
        "has_video":    tg["has_video"],
        "has_link":     tg["has_link"],
        "has_poll":     tg["has_poll"],
        "is_ad":        tg["is_ad"],
        "link":         tg["message_link"].fillna(tg["source_link"]),
        # оставляем важные TG-поля
        "tg_channel_id":       tg["channel_id"],
        "tg_channel_username": tg["channel_username"],
        "tg_is_forwarded":     tg["is_forwarded"],
        "tg_fwd_src_id":       tg["fwd_src_id"],
        "tg_fwd_src_name":     tg["fwd_src_name"],
        "tg_fwd_src_msg_id":   tg["fwd_src_msg_id"],
        "image_url":           tg["image_url"],
    })
    return out

def collect_snapshot_cols(df):
    snaps = {}         # {date: {"likes": col, "comments": col, "reposts": col}}
    collected = {}     # {date: col}
    for c in df.columns:
        m = SNAP_PAT.match(str(c))
        if m:
            metric, d = m.group(1), m.group(2)
            snaps.setdefault(d, {})[metric] = c
            continue
        k = COLLECTED_PAT.match(str(c))
        if k:
            d = k.group(1)
            collected[d] = c
    return snaps, collected

def attach_snapshots(base, raw, snaps, collected):
    out = base.copy()
    for d, mapping in sorted(snaps.items()):
        for metric, col in mapping.items():
            out[f"{metric}_{d}"] = pd.to_numeric(raw[col], errors="coerce")
        # если есть collected_at_* — добавим текстовым столбцом
        ccol = collected.get(d)
        if ccol:
            out[f"collected_at_{d}"] = raw[ccol].astype(str)
    return out

def compute_er(wide):
    # er_<date> = reposts_<date> / followers
    dates = []
    for c in wide.columns:
        m = re.match(r"^reposts_(\d{4}-\d{2}-\d{2})$", c)
        if m: dates.append(m.group(1))
    dates = sorted(set(dates))
    for d in dates:
        num = pd.to_numeric(wide.get(f"reposts_{d}"), errors="coerce")
        den = pd.to_numeric(wide.get("followers"), errors="coerce")
        wide[f"er_{d}"] = np.where((den > 0) & num.notna(), num/den, np.nan)
    # итоговый engagement_rate = по самой поздней дате с ненулевым значением
    if dates:
        # идём по датам в хронологическом порядке, перезаписывая на более поздние
        latest = pd.Series(np.nan, index=wide.index, dtype=float)
        for d in dates:
            v = wide[f"er_{d}"]
            latest = np.where(~np.isnan(v), v, latest)
        wide["engagement_rate"] = latest
    return dates, wide

# --- загрузка ---
tg_raw = read_df(TG_CSV)
vk_raw = read_df(VK_CSV)

frames = []

# TG
if not tg_raw.empty:
    tg_base = base_block_tg(tg_raw)
    tg_snaps, tg_collected = collect_snapshot_cols(tg_raw)
    tg_wide = attach_snapshots(tg_base, tg_raw, tg_snaps, tg_collected)
    frames.append(tg_wide)

# VK
if not vk_raw.empty:
    vk_base = base_block_vk(vk_raw)
    vk_snaps, vk_collected = collect_snapshot_cols(vk_raw)
    vk_wide = attach_snapshots(vk_base, vk_raw, vk_snaps, vk_collected)
    frames.append(vk_wide)

# объединение
wide = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

# ER
snapshot_dates, wide = compute_er(wide)

# типобезопасность перед parquet
# строки — явно как string; булевые — в bool; числа — numeric
for col in ["platform","source_name","source_link","published_at","text","link",
            "tg_channel_username","tg_fwd_src_name","image_url"]:
    if col in wide.columns:
        wide[col] = wide[col].astype("string")

for col in ["has_photo","has_video","has_link","has_poll","is_ad","tg_is_forwarded"]:
    if col in wide.columns:
        # аккуратно приводим к bool через map
        wide[col] = wide[col].map(lambda x: bool(x) if pd.notna(x) else pd.NA)

num_like = [c for c in wide.columns if re.match(r"^(likes|comments|reposts|er)_\d{4}-\d{2}-\d{2}$", c)]
for col in ["followers"] + num_like + ["engagement_rate"]:
    if col in wide.columns:
        wide[col] = pd.to_numeric(wide[col], errors="coerce")

# сохранение
wide.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
wide.to_parquet(OUT_PQ, index=False)

# отчёт
dates_found = sorted({m.group(2) for col in wide.columns for m in [SNAP_PAT.match(col)] if m})
print("Saved:", OUT_CSV)
print("Saved:", OUT_PQ)
print("Shape:", wide.shape)
print("Snapshot dates:", dates_found)
print(wide.head(3))
print(wide.info())



Saved: /content/social_wide.csv
Saved: /content/social_wide.parquet
Shape: (21361, 32)
Snapshot dates: ['2025-06-07', '2025-08-15']
  platform             source_name                 source_link  followers  \
0       tg  Москва с огоньком Live  https://t.me/moscowtoplive     101489   
1       tg  Москва с огоньком Live  https://t.me/moscowtoplive     101489   
2       tg  Москва с огоньком Live  https://t.me/moscowtoplive     101489   

   post_id               published_at  \
0    32607  2025-08-14T09:02:35+00:00   
1    32606  2025-08-14T08:41:34+00:00   
2    32605  2025-08-14T07:55:53+00:00   

                                                text  has_photo  has_video  \
0  Муром: 150 миллионов рублей на подсветку моста...      False       True   
1  Туристка из РФ погибла в результате ДТП в Егип...      False      False   
2  Слова «миллион» и «миллиард» теперь можно прои...      False      False   

   has_link  ...  reposts_2025-08-15 collected_at_2025-08-15 vk_group_id  \
0    

In [ ]:
# WIDE из TG+VK + фиксация TG-снапшотов, ER по репостам/подписчикам
# + включены базовые колонки likes/comments/reposts и text_len (TG; для VK посчитан)
import os, re, unicodedata
import numpy as np
import pandas as pd

# --- пути ---
TG_CSV = "/content/tg_merged.csv"
VK_CSV = "/content/vk_posts.csv"
OUT_CSV = "/content/social_wide.csv"
OUT_PQ  = "/content/social_wide.parquet"

# --- паттерны ---
SNAP_PAT       = re.compile(r"^(?:original_)?(likes|comments|reposts)_(\d{4}-\d{2}-\d{2})$")
COLLECTED_PAT  = re.compile(r"^(?:original_)?collected_at_(\d{4}-\d{2}-\d{2})$")
REPOSTS_DATERE = re.compile(r"^reposts_(\d{4}-\d{2}-\d{2})$")

# --- чтение и нормализация заголовков ---
def read_df(path):
    if not os.path.exists(path):
        return pd.DataFrame()
    return pd.read_parquet(path) if path.endswith(".parquet") else pd.read_csv(path, dtype=str, low_memory=False)

def norm_cols(df):
    cols = []
    for c in df.columns:
        s = unicodedata.normalize("NFKC", str(c))
        s = (s.replace("\u2010","-").replace("\u2011","-").replace("\u2012","-")
               .replace("\u2013","-").replace("\u2014","-").replace("\u2212","-")).strip()
        cols.append(s)
    df.columns = cols
    return df

tg_raw = norm_cols(read_df(TG_CSV))
vk_raw = norm_cols(read_df(VK_CSV))

# --- поля, которые должны быть ---
VK_KEEP = [
    "group_id","post_id","from_id","text","date","initial_date","likes",
    "comments","reposts","image_url","followers","source_name","source_link",
    "is_ad","has_photo","has_video","has_link","has_poll"
]
TG_KEEP = [
    "platform","source_name","source_link","followers","channel_id","channel_username",
    "message_id","date","initial_date","text","text_len","likes","comments","reposts",
    "has_photo","has_video","has_link","has_poll","is_ad","image_url","message_link",
    "is_forwarded","fwd_src_id","fwd_src_name","fwd_src_msg_id"
]

def ensure_cols(df, cols):
    for c in cols:
        if c not in df.columns:
            df[c] = pd.NA
    return df

vk_raw = ensure_cols(vk_raw, VK_KEEP)
tg_raw = ensure_cols(tg_raw, TG_KEEP)

# --- базовые блоки: добавляем базовые likes/comments/reposts и text_len ---
# VK: считаем text_len, т.к. его нет в исходнике
vk_text_len = vk_raw["text"].fillna("").astype(str).str.len()

vk_base = pd.DataFrame({
    "platform":     "vk",
    "source_name":  vk_raw["source_name"],
    "source_link":  vk_raw["source_link"],
    "followers":    pd.to_numeric(vk_raw["followers"], errors="coerce"),
    "post_id":      pd.to_numeric(vk_raw["post_id"], errors="coerce").astype("Int64"),
    "published_at": vk_raw["initial_date"].fillna(vk_raw["date"]),
    "text":         vk_raw["text"],
    "text_len":     pd.to_numeric(vk_text_len, errors="coerce").astype("Int64"),
    "has_photo":    vk_raw["has_photo"],
    "has_video":    vk_raw["has_video"],
    "has_link":     vk_raw["has_link"],
    "has_poll":     vk_raw["has_poll"],
    "is_ad":        vk_raw["is_ad"],
    "link":         vk_raw["source_link"],
    "vk_group_id":  vk_raw["group_id"],
    "vk_from_id":   vk_raw["from_id"],
    "image_url":    vk_raw["image_url"],
    # базовые метрики из исходника VK
    "likes":        pd.to_numeric(vk_raw["likes"], errors="coerce"),
    "comments":     pd.to_numeric(vk_raw["comments"], errors="coerce"),
    "reposts":      pd.to_numeric(vk_raw["reposts"], errors="coerce"),
})

# TG: берём готовый text_len, если нет — считаем
tg_text_len = tg_raw["text_len"]
if tg_text_len.isna().all():
    tg_text_len = tg_raw["text"].fillna("").astype(str).str.len()

tg_base = pd.DataFrame({
    "platform":     "tg",
    "source_name":  tg_raw["source_name"],
    "source_link":  tg_raw["source_link"],
    "followers":    pd.to_numeric(tg_raw["followers"], errors="coerce"),
    "post_id":      pd.to_numeric(tg_raw["message_id"], errors="coerce").astype("Int64"),
    "published_at": tg_raw["initial_date"].fillna(tg_raw["date"]),
    "text":         tg_raw["text"],
    "text_len":     pd.to_numeric(tg_text_len, errors="coerce").astype("Int64"),
    "has_photo":    tg_raw["has_photo"],
    "has_video":    tg_raw["has_video"],
    "has_link":     tg_raw["has_link"],
    "has_poll":     tg_raw["has_poll"],
    "is_ad":        tg_raw["is_ad"],
    "link":         tg_raw["message_link"].fillna(tg_raw["source_link"]),
    "tg_channel_id":       pd.to_numeric(tg_raw["channel_id"], errors="coerce").astype("Int64"),
    "tg_channel_username": tg_raw["channel_username"],
    "tg_is_forwarded":     tg_raw["is_forwarded"],
    "tg_fwd_src_id":       pd.to_numeric(tg_raw["fwd_src_id"], errors="coerce").astype("Int64"),
    "tg_fwd_src_name":     tg_raw["fwd_src_name"],
    "tg_fwd_src_msg_id":   pd.to_numeric(tg_raw["fwd_src_msg_id"], errors="coerce").astype("Int64"),
    "image_url":           tg_raw["image_url"],
    # базовые метрики из исходника TG (likes/comments/reposts)
    "likes":        pd.to_numeric(tg_raw["likes"], errors="coerce"),
    "comments":     pd.to_numeric(tg_raw["comments"], errors="coerce"),
    "reposts":      pd.to_numeric(tg_raw["reposts"], errors="coerce"),
})

# --- собрать списки снапшотов/collected_at ---
def collect_snapshots(df):
    snaps = {}; collected = {}
    for c in df.columns:
        m = SNAP_PAT.match(c)
        if m:
            metric, d = m.group(1), m.group(2)
            snaps.setdefault(d, {})[metric] = c
            continue
        k = COLLECTED_PAT.match(c)
        if k:
            d = k.group(1); collected[d] = c
    return snaps, collected

vk_snaps, vk_collected = collect_snapshots(vk_raw)
tg_snaps, tg_collected = collect_snapshots(tg_raw)

# --- прикрутить снапшоты к базовым ---
def attach_snaps(base, raw, snaps, collected):
    out = base.copy()
    for d, mapping in sorted(snaps.items()):
        for metric, col in mapping.items():
            out[f"{metric}_{d}"] = pd.to_numeric(raw[col], errors="coerce")
        if d in collected:
            out[f"collected_at_{d}"] = raw[collected[d]].astype(str)
    return out

vk_wide = attach_snaps(vk_base, vk_raw, vk_snaps, vk_collected) if not vk_base.empty else pd.DataFrame()
tg_wide = attach_snaps(tg_base, tg_raw, tg_snaps, tg_collected) if not tg_base.empty else pd.DataFrame()

# --- явная подтяжка TG-снапшотов (закрыть пропуски) ---
tg_slim_cols = ["message_id"] + sorted([c for c in tg_raw.columns if SNAP_PAT.match(c) or COLLECTED_PAT.match(c)])
tg_slim = tg_raw[tg_slim_cols].copy()
tg_slim["message_id"] = pd.to_numeric(tg_slim["message_id"], errors="coerce").astype("Int64")

sw_tg = tg_wide.merge(
    tg_slim, left_on="post_id", right_on="message_id", how="left", suffixes=("","__tg")
)
for c in tg_slim_cols:
    if c == "message_id":
        continue
    if (c in sw_tg.columns) and (c + "__tg" in sw_tg.columns):
        fill_mask = sw_tg[c].isna() & sw_tg[c + "__tg"].notna()
        sw_tg.loc[fill_mask, c] = pd.to_numeric(sw_tg.loc[fill_mask, c + "__tg"], errors="coerce")
sw_tg.drop(columns=[c for c in sw_tg.columns if c.endswith("__tg")] + ["message_id"], inplace=True, errors="ignore")

# --- объединение ---
wide = pd.concat([vk_wide, sw_tg], ignore_index=True, sort=False)

# --- ER по датам и engagement_rate ---
dates = sorted({REPOSTS_DATERE.match(c).group(1) for c in wide.columns if REPOSTS_DATERE.match(c)})
for d in dates:
    num = pd.to_numeric(wide.get(f"reposts_{d}"), errors="coerce")
    den = pd.to_numeric(wide.get("followers"), errors="coerce")
    wide[f"er_{d}"] = np.where((den > 0) & num.notna(), num/den, np.nan)

wide["engagement_rate"] = np.nan
for d in dates:  # по возрастанию дат: поздние перезаписывают
    erd = pd.to_numeric(wide.get(f"er_{d}"), errors="coerce")
    wide["engagement_rate"] = np.where(erd.notna(), erd, wide["engagement_rate"])

# --- типы перед parquet ---
for col in ["platform","source_name","source_link","published_at","text","link",
            "tg_channel_username","tg_fwd_src_name","image_url"]:
    if col in wide.columns:
        wide[col] = wide[col].astype("string")

for col in ["has_photo","has_video","has_link","has_poll","is_ad","tg_is_forwarded"]:
    if col in wide.columns:
        wide[col] = wide[col].map(lambda x: bool(x) if pd.notna(x) and str(x).lower() not in ("nan","none") else pd.NA)

num_like = [c for c in wide.columns if re.match(r"^(likes|comments|reposts|er)_\d{4}-\d{2}-\d{2}$", c)]
num_base = ["likes","comments","reposts","followers","vk_group_id","vk_from_id","tg_channel_id","tg_fwd_src_id","tg_fwd_src_msg_id","text_len","engagement_rate"]
for col in num_base + num_like:
    if col in wide.columns:
        wide[col] = pd.to_numeric(wide[col], errors="coerce")

wide["post_id"] = pd.to_numeric(wide["post_id"], errors="coerce").astype("Int64")

# --- сохранение ---
wide.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
wide.to_parquet(OUT_PQ, index=False)

# --- отчёт ---
print("Saved:", OUT_CSV)
print("Saved:", OUT_PQ)
print("Shape:", wide.shape)
print("Snapshot dates:", dates)
print("TG snaps non-null:", wide.loc[wide["platform"]=="tg", ["likes_2025-08-15","comments_2025-08-15","reposts_2025-08-15"]].notna().sum().to_dict())
print("VK base metrics non-null:", wide.loc[wide["platform"]=="vk", ["likes","comments","reposts"]].notna().sum().to_dict())
print("TG base metrics non-null:", wide.loc[wide["platform"]=="tg", ["likes","comments","reposts","text_len"]].notna().sum().to_dict())



Saved: /content/social_wide.csv
Saved: /content/social_wide.parquet
Shape: (24735, 40)
Snapshot dates: ['2025-06-07', '2025-08-15']
TG snaps non-null: {'likes_2025-08-15': 7207, 'comments_2025-08-15': 4233, 'reposts_2025-08-15': 12219}
VK base metrics non-null: {'likes': 12457, 'comments': 12457, 'reposts': 12457}
TG base metrics non-null: {'likes': 7198, 'comments': 4221, 'reposts': 12234, 'text_len': 12278}


In [ ]:
# WIDE из TG+VK. Фикс мерджа TG по составному ключу. ER и engagement_rate.

import os, re, unicodedata
import numpy as np
import pandas as pd

# --- пути ---
TG_CSV = "/content/tg_merged.csv"
VK_CSV = "/content/vk_posts.csv"
OUT_CSV = "/content/social_wide4.csv"
OUT_PQ  = "/content/social_wide4.parquet"

# --- паттерны ---
SNAP_PAT       = re.compile(r"^(?:original_)?(likes|comments|reposts)_(\d{4}-\d{2}-\d{2})$")
COLLECTED_PAT  = re.compile(r"^(?:original_)?collected_at_(\d{4}-\d{2}-\d{2})$")
REPOSTS_DATERE = re.compile(r"^reposts_(\d{4}-\d{2}-\d{2})$")

# --- чтение и нормализация заголовков ---
def read_df(path):
    if not os.path.exists(path):
        return pd.DataFrame()
    return pd.read_parquet(path) if path.endswith(".parquet") else pd.read_csv(path, dtype=str, low_memory=False)

def norm_cols(df):
    cols = []
    for c in df.columns:
        s = unicodedata.normalize("NFKC", str(c))
        s = (s.replace("\u2010","-").replace("\u2011","-").replace("\u2012","-")
               .replace("\u2013","-").replace("\u2014","-").replace("\u2212","-")).strip()
        cols.append(s)
    df.columns = cols
    return df

tg_raw = norm_cols(read_df(TG_CSV))
vk_raw = norm_cols(read_df(VK_CSV))

# --- обязательные поля ---
VK_KEEP = [
    "group_id","post_id","from_id","text","date","initial_date","likes",
    "comments","reposts","image_url","followers","source_name","source_link",
    "is_ad","has_photo","has_video","has_link","has_poll"
]
TG_KEEP = [
    "platform","source_name","source_link","followers","channel_id","channel_username",
    "message_id","date","initial_date","text","text_len","likes","comments","reposts",
    "has_photo","has_video","has_link","has_poll","is_ad","image_url","message_link",
    "is_forwarded","fwd_src_id","fwd_src_name","fwd_src_msg_id"
]

def ensure_cols(df, cols):
    for c in cols:
        if c not in df.columns:
            df[c] = pd.NA
    return df

vk_raw = ensure_cols(vk_raw, VK_KEEP)
tg_raw = ensure_cols(tg_raw, TG_KEEP)

# --- базовые блоки + базовые метрики + text_len ---
vk_text_len = vk_raw["text"].fillna("").astype(str).str.len()

vk_base = pd.DataFrame({
    "platform":     "vk",
    "source_name":  vk_raw["source_name"],
    "source_link":  vk_raw["source_link"],
    "followers":    pd.to_numeric(vk_raw["followers"], errors="coerce"),
    "post_id":      pd.to_numeric(vk_raw["post_id"], errors="coerce").astype("Int64"),
    "published_at": vk_raw["initial_date"].fillna(vk_raw["date"]),
    "text":         vk_raw["text"],
    "text_len":     pd.to_numeric(vk_text_len, errors="coerce").astype("Int64"),
    "has_photo":    vk_raw["has_photo"],
    "has_video":    vk_raw["has_video"],
    "has_link":     vk_raw["has_link"],
    "has_poll":     vk_raw["has_poll"],
    "is_ad":        vk_raw["is_ad"],
    "link":         vk_raw["source_link"],
    "vk_group_id":  vk_raw["group_id"],
    "vk_from_id":   vk_raw["from_id"],
    "image_url":    vk_raw["image_url"],
    "likes":        pd.to_numeric(vk_raw["likes"], errors="coerce"),
    "comments":     pd.to_numeric(vk_raw["comments"], errors="coerce"),
    "reposts":      pd.to_numeric(vk_raw["reposts"], errors="coerce"),
})

tg_text_len = tg_raw["text_len"]
if tg_text_len.isna().all():
    tg_text_len = tg_raw["text"].fillna("").astype(str).str.len()

tg_base = pd.DataFrame({
    "platform":     "tg",
    "source_name":  tg_raw["source_name"],
    "source_link":  tg_raw["source_link"],
    "followers":    pd.to_numeric(tg_raw["followers"], errors="coerce"),
    "post_id":      pd.to_numeric(tg_raw["message_id"], errors="coerce").astype("Int64"),
    "published_at": tg_raw["initial_date"].fillna(tg_raw["date"]),
    "text":         tg_raw["text"],
    "text_len":     pd.to_numeric(tg_text_len, errors="coerce").astype("Int64"),
    "has_photo":    tg_raw["has_photo"],
    "has_video":    tg_raw["has_video"],
    "has_link":     tg_raw["has_link"],
    "has_poll":     tg_raw["has_poll"],
    "is_ad":        tg_raw["is_ad"],
    "link":         tg_raw["message_link"].fillna(tg_raw["source_link"]),
    "tg_channel_id":       pd.to_numeric(tg_raw["channel_id"], errors="coerce").astype("Int64"),
    "tg_channel_username": tg_raw["channel_username"].astype("string"),
    "tg_is_forwarded":     tg_raw["is_forwarded"],
    "tg_fwd_src_id":       pd.to_numeric(tg_raw["fwd_src_id"], errors="coerce").astype("Int64"),
    "tg_fwd_src_name":     tg_raw["fwd_src_name"],
    "tg_fwd_src_msg_id":   pd.to_numeric(tg_raw["fwd_src_msg_id"], errors="coerce").astype("Int64"),
    "image_url":           tg_raw["image_url"],
    "likes":        pd.to_numeric(tg_raw["likes"], errors="coerce"),
    "comments":     pd.to_numeric(tg_raw["comments"], errors="coerce"),
    "reposts":      pd.to_numeric(tg_raw["reposts"], errors="coerce"),
})

# --- сбор списков снапшотов/collected_at ---
def collect_snapshots(df):
    snaps = {}; collected = {}
    for c in df.columns:
        m = SNAP_PAT.match(c)
        if m:
            metric, d = m.group(1), m.group(2)
            snaps.setdefault(d, {})[metric] = c
            continue
        k = COLLECTED_PAT.match(c)
        if k:
            d = k.group(1); collected[d] = c
    return snaps, collected

vk_snaps, vk_collected = collect_snapshots(vk_raw)
tg_snaps, tg_collected = collect_snapshots(tg_raw)

# --- прикрутить снапшоты к базовым ---
def attach_snaps(base, raw, snaps, collected):
    out = base.copy()
    for d, mapping in sorted(snaps.items()):
        for metric, col in mapping.items():
            out[f"{metric}_{d}"] = pd.to_numeric(raw[col], errors="coerce")
        if d in collected:
            out[f"collected_at_{d}"] = raw[collected[d]].astype(str)
    return out

vk_wide = attach_snaps(vk_base, vk_raw, vk_snaps, vk_collected) if not vk_base.empty else pd.DataFrame()
tg_wide = attach_snaps(tg_base, tg_raw, tg_snaps, tg_collected) if not tg_base.empty else pd.DataFrame()

# --- ЯВНАЯ подтяжка TG-снапшотов по составному ключу (post_id + tg_channel_id + tg_channel_username) ---
tg_slim_cols = ["message_id", "channel_id", "channel_username"] + sorted(
    [c for c in tg_raw.columns if SNAP_PAT.match(c) or COLLECTED_PAT.match(c)]
)
tg_slim = tg_raw[tg_slim_cols].copy()
tg_slim["message_id"] = pd.to_numeric(tg_slim["message_id"], errors="coerce").astype("Int64")
tg_slim["channel_id"] = pd.to_numeric(tg_slim["channel_id"], errors="coerce").astype("Int64")
tg_slim["channel_username"] = tg_slim["channel_username"].astype("string")

sw_tg = tg_wide.merge(
    tg_slim,
    left_on=["post_id", "tg_channel_id", "tg_channel_username"],
    right_on=["message_id", "channel_id", "channel_username"],
    how="left",
    suffixes=("", "__tg")
)

to_fill = [c for c in tg_slim.columns if c not in ("message_id","channel_id","channel_username")]
for c in to_fill:
    src = c + "__tg"
    if c in sw_tg.columns and src in sw_tg.columns:
        fill_mask = sw_tg[c].isna() & sw_tg[src].notna()
        sw_tg.loc[fill_mask, c] = pd.to_numeric(sw_tg.loc[fill_mask, src], errors="coerce")

sw_tg.drop(columns=[c for c in sw_tg.columns if c.endswith("__tg")] + ["message_id","channel_id","channel_username"],
           inplace=True, errors="ignore")

# --- объединение TG+VK ---
wide = pd.concat([vk_wide, sw_tg], ignore_index=True, sort=False)

# --- ER по датам ---
dates = sorted({REPOSTS_DATERE.match(c).group(1) for c in wide.columns if REPOSTS_DATERE.match(c)})
for d in dates:
    num = pd.to_numeric(wide.get(f"reposts_{d}"), errors="coerce")
    den = pd.to_numeric(wide.get("followers"), errors="coerce")
    wide[f"er_{d}"] = np.where((den > 0) & num.notna(), num/den, np.nan)

# --- engagement_rate из базового reposts/followers ---
wide["engagement_rate"] = np.where(
    (pd.to_numeric(wide.get("followers"), errors="coerce") > 0) &
    pd.to_numeric(wide.get("reposts"), errors="coerce").notna(),
    pd.to_numeric(wide.get("reposts"), errors="coerce") / pd.to_numeric(wide.get("followers"), errors="coerce"),
    np.nan
).astype(float)

# --- безопасная подстановка: reposts_2025-08-15 <- reposts, только где NaN ---
if "reposts_2025-08-15" in wide.columns and "reposts" in wide.columns:
    empty_snap = wide["reposts_2025-08-15"].isna() & wide["reposts"].notna()
    wide.loc[empty_snap, "reposts_2025-08-15"] = pd.to_numeric(wide.loc[empty_snap, "reposts"], errors="coerce")

# --- er_final: TG = reposts_2025-08-15 / followers; VK = reposts_2025-06-07 / followers ---
for c in ["followers","reposts_2025-08-15","reposts_2025-06-07"]:
    if c in wide.columns:
        wide[c] = pd.to_numeric(wide[c], errors="coerce")

wide["er_final"] = np.nan
m_tg = (wide["platform"] == "tg") & wide["reposts_2025-08-15"].notna() & (wide["followers"] > 0)
wide.loc[m_tg, "er_final"] = wide.loc[m_tg, "reposts_2025-08-15"] / wide.loc[m_tg, "followers"]

m_vk = (wide["platform"] == "vk") & wide["reposts_2025-06-07"].notna() & (wide["followers"] > 0)
wide.loc[m_vk, "er_final"] = wide.loc[m_vk, "reposts_2025-06-07"] / wide.loc[m_vk, "followers"]

wide["er_final"] = pd.to_numeric(wide["er_final"], errors="coerce")

# --- типы перед parquet ---
for col in ["platform","source_name","source_link","published_at","text","link",
            "tg_channel_username","tg_fwd_src_name","image_url"]:
    if col in wide.columns:
        wide[col] = wide[col].astype("string")

for col in ["has_photo","has_video","has_link","has_poll","is_ad","tg_is_forwarded"]:
    if col in wide.columns:
        wide[col] = wide[col].map(lambda x: bool(x) if pd.notna(x) and str(x).lower() not in ("nan","none") else pd.NA)

num_like = [c for c in wide.columns if re.match(r"^(likes|comments|reposts|er)_\d{4}-\d{2}-\d{2}$", c)]
num_base = ["likes","comments","reposts","followers","vk_group_id","vk_from_id","tg_channel_id",
            "tg_fwd_src_id","tg_fwd_src_msg_id","text_len","engagement_rate","er_final"]
for col in num_base + num_like:
    if col in wide.columns:
        wide[col] = pd.to_numeric(wide[col], errors="coerce")

wide["post_id"] = pd.to_numeric(wide["post_id"], errors="coerce").astype("Int64")

# --- сохранение ---
wide.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
try:
    wide.to_parquet(OUT_PQ, index=False)
except Exception as e:
    print("Parquet save skipped:", e)

# --- отчёт ---
print("Saved:", OUT_CSV)
print("Saved:", OUT_PQ)
print("Shape:", wide.shape)
print("Snapshot dates:", dates)
print("TG snaps non-null:", wide.loc[wide["platform"]=="tg",
      ["likes_2025-08-15","comments_2025-08-15","reposts_2025-08-15"]].notna().sum().to_dict())
print("VK base metrics non-null:", wide.loc[wide["platform"]=="vk",
      ["likes","comments","reposts"]].notna().sum().to_dict())
print("TG base metrics non-null:", wide.loc[wide["platform"]=="tg",
      ["likes","comments","reposts","text_len"]].notna().sum().to_dict())
print("er_final non-null total:", int(wide["er_final"].notna().sum()))
print(wide[["platform","tg_channel_username","post_id","followers",
            "reposts","reposts_2025-06-07","reposts_2025-08-15","er_final","engagement_rate"]].tail(10))


Saved: /content/social_wide4.csv
Saved: /content/social_wide4.parquet
Shape: (21361, 41)
Snapshot dates: ['2025-06-07', '2025-08-15']
TG snaps non-null: {'likes_2025-08-15': 5004, 'comments_2025-08-15': 2839, 'reposts_2025-08-15': 8877}
VK base metrics non-null: {'likes': 12457, 'comments': 12457, 'reposts': 12457}
TG base metrics non-null: {'likes': 5011, 'comments': 2845, 'reposts': 8877, 'text_len': 8904}
er_final non-null total: 21334
      platform tg_channel_username  post_id  followers  reposts  \
21351       tg            bankoffo    36635     180022    188.0   
21352       tg            bankoffo    36634     180022    188.0   
21353       tg            bankoffo    36633     180022    188.0   
21354       tg            bankoffo    36632     180022    203.0   
21355       tg            bankoffo    36631     180022    136.0   
21356       tg            bankoffo    36630     180022    384.0   
21357       tg            bankoffo    36629     180022    200.0   
21358       tg       

In [ ]:
# === Social WIDE build: TG+VK, безопасный мердж без «чистки CSV» ===
import os, re, unicodedata
import numpy as np
import pandas as pd
import csv

# --- пути ---
TG_CSV = "/content/tg_merged.csv"
VK_CSV = "/content/vk_posts.csv"
OUT_CSV = "/content/social_wide_5.csv"
OUT_PQ  = "/content/social_wide_5.parquet"

# --- паттерны ---
SNAP_PAT       = re.compile(r"^(?:original_)?(likes|comments|reposts)_(\d{4}-\d{2}-\d{2})$")
COLLECTED_PAT  = re.compile(r"^(?:original_)?collected_at_(\d{4}-\d{2}-\d{2})$")
REPOSTS_DATERE = re.compile(r"^reposts_(\d{4}-\d{2}-\d{2})$")

# --- чтение и нормализация заголовков ---
def read_df(path):
    if not os.path.exists(path):
        return pd.DataFrame()
    # pandas сам обработает кавычки и переносы строк внутри ячеек
    return pd.read_parquet(path) if path.endswith(".parquet") else pd.read_csv(
        path, dtype=str, low_memory=False
    )

def norm_cols(df):
    cols = []
    for c in df.columns:
        s = unicodedata.normalize("NFKC", str(c))
        s = (s.replace("\u2010","-").replace("\u2011","-").replace("\u2012","-")
               .replace("\u2013","-").replace("\u2014","-").replace("\u2212","-")).strip()
        cols.append(s)
    df.columns = cols
    return df

tg_raw = norm_cols(read_df(TG_CSV))
vk_raw = norm_cols(read_df(VK_CSV))

# --- обязательные поля ---
VK_KEEP = [
    "group_id","post_id","from_id","text","date","initial_date","likes",
    "comments","reposts","image_url","followers","source_name","source_link",
    "is_ad","has_photo","has_video","has_link","has_poll"
]
TG_KEEP = [
    "platform","source_name","source_link","followers","channel_id","channel_username",
    "message_id","date","initial_date","text","text_len","likes","comments","reposts",
    "has_photo","has_video","has_link","has_poll","is_ad","image_url","message_link",
    "is_forwarded","fwd_src_id","fwd_src_name","fwd_src_msg_id"
]

def ensure_cols(df, cols):
    for c in cols:
        if c not in df.columns:
            df[c] = pd.NA
    return df

vk_raw = ensure_cols(vk_raw, VK_KEEP)
tg_raw = ensure_cols(tg_raw, TG_KEEP)

# --- базовые блоки + метрики + text_len ---
vk_text_len = vk_raw["text"].fillna("").astype(str).str.len()

vk_base = pd.DataFrame({
    "platform":     "vk",
    "source_name":  vk_raw["source_name"],
    "source_link":  vk_raw["source_link"],
    "followers":    pd.to_numeric(vk_raw["followers"], errors="coerce"),
    "post_id":      pd.to_numeric(vk_raw["post_id"], errors="coerce").astype("Int64"),
    "published_at": vk_raw["initial_date"].fillna(vk_raw["date"]),
    "text":         vk_raw["text"],
    "text_len":     pd.to_numeric(vk_text_len, errors="coerce").astype("Int64"),
    "has_photo":    vk_raw["has_photo"],
    "has_video":    vk_raw["has_video"],
    "has_link":     vk_raw["has_link"],
    "has_poll":     vk_raw["has_poll"],
    "is_ad":        vk_raw["is_ad"],
    "link":         vk_raw["source_link"],
    "vk_group_id":  vk_raw["group_id"],
    "vk_from_id":   vk_raw["from_id"],
    "image_url":    vk_raw["image_url"],
    "likes":        pd.to_numeric(vk_raw["likes"], errors="coerce"),
    "comments":     pd.to_numeric(vk_raw["comments"], errors="coerce"),
    "reposts":      pd.to_numeric(vk_raw["reposts"], errors="coerce"),
})

tg_text_len = tg_raw["text_len"]
if tg_text_len.isna().all():
    tg_text_len = tg_raw["text"].fillna("").astype(str).str.len()

tg_base = pd.DataFrame({
    "platform":     "tg",
    "source_name":  tg_raw["source_name"],
    "source_link":  tg_raw["source_link"],
    "followers":    pd.to_numeric(tg_raw["followers"], errors="coerce"),
    "post_id":      pd.to_numeric(tg_raw["message_id"], errors="coerce").astype("Int64"),
    "published_at": tg_raw["initial_date"].fillna(tg_raw["date"]),
    "text":         tg_raw["text"],
    "text_len":     pd.to_numeric(tg_text_len, errors="coerce").astype("Int64"),
    "has_photo":    tg_raw["has_photo"],
    "has_video":    tg_raw["has_video"],
    "has_link":     tg_raw["has_link"],
    "has_poll":     tg_raw["has_poll"],
    "is_ad":        tg_raw["is_ad"],
    "link":         tg_raw["message_link"].fillna(tg_raw["source_link"]),
    "tg_channel_id":       pd.to_numeric(tg_raw["channel_id"], errors="coerce").astype("Int64"),
    "tg_channel_username": tg_raw["channel_username"].astype("string"),
    "tg_is_forwarded":     tg_raw["is_forwarded"],
    "tg_fwd_src_id":       pd.to_numeric(tg_raw["fwd_src_id"], errors="coerce").astype("Int64"),
    "tg_fwd_src_name":     tg_raw["fwd_src_name"],
    "tg_fwd_src_msg_id":   pd.to_numeric(tg_raw["fwd_src_msg_id"], errors="coerce").astype("Int64"),
    "image_url":           tg_raw["image_url"],
    "likes":        pd.to_numeric(tg_raw["likes"], errors="coerce"),
    "comments":     pd.to_numeric(tg_raw["comments"], errors="coerce"),
    "reposts":      pd.to_numeric(tg_raw["reposts"], errors="coerce"),
})

# --- снапшоты/collected_at ---
def collect_snapshots(df):
    snaps = {}; collected = {}
    for c in df.columns:
        m = SNAP_PAT.match(c)
        if m:
            metric, d = m.group(1), m.group(2)
            snaps.setdefault(d, {})[metric] = c
            continue
        k = COLLECTED_PAT.match(c)
        if k:
            d = k.group(1); collected[d] = c
    return snaps, collected

vk_snaps, vk_collected = collect_snapshots(vk_raw)
tg_snaps, tg_collected = collect_snapshots(tg_raw)

def attach_snaps(base, raw, snaps, collected):
    out = base.copy()
    for d, mapping in sorted(snaps.items()):
        for metric, col in mapping.items():
            out[f"{metric}_{d}"] = pd.to_numeric(raw[col], errors="coerce")
        if d in collected:
            out[f"collected_at_{d}"] = raw[collected[d]].astype(str)
    return out

vk_wide = attach_snaps(vk_base, vk_raw, vk_snaps, vk_collected) if not vk_base.empty else pd.DataFrame()
tg_wide = attach_snaps(tg_base, tg_raw, tg_snaps, tg_collected) if not tg_base.empty else pd.DataFrame()

# --- TG: явная подтяжка снапшотов по составному ключу ---
tg_slim_cols = ["message_id", "channel_id", "channel_username"] + sorted(
    [c for c in tg_raw.columns if SNAP_PAT.match(c) or COLLECTED_PAT.match(c)]
)
tg_slim = tg_raw[tg_slim_cols].copy()
tg_slim["message_id"] = pd.to_numeric(tg_slim["message_id"], errors="coerce").astype("Int64")
tg_slim["channel_id"] = pd.to_numeric(tg_slim["channel_id"], errors="coerce").astype("Int64")
tg_slim["channel_username"] = tg_slim["channel_username"].astype("string")

sw_tg = tg_wide.merge(
    tg_slim,
    left_on=["post_id", "tg_channel_id", "tg_channel_username"],
    right_on=["message_id", "channel_id", "channel_username"],
    how="left",
    suffixes=("", "__tg")
)

to_fill = [c for c in tg_slim.columns if c not in ("message_id","channel_id","channel_username")]
for c in to_fill:
    src = c + "__tg"
    if c in sw_tg.columns and src in sw_tg.columns:
        mask = sw_tg[c].isna() & sw_tg[src].notna()
        sw_tg.loc[mask, c] = pd.to_numeric(sw_tg.loc[mask, src], errors="coerce")

sw_tg.drop(columns=[c for c in sw_tg.columns if c.endswith("__tg")] + ["message_id","channel_id","channel_username"],
           inplace=True, errors="ignore")

# --- объединение TG+VK ---
sw = pd.concat([vk_wide, sw_tg], ignore_index=True, sort=False)

# --- ER по датам ---
dates = sorted({REPOSTS_DATERE.match(c).group(1) for c in sw.columns if REPOSTS_DATERE.match(c)})
for d in dates:
    num = pd.to_numeric(sw.get(f"reposts_{d}"), errors="coerce")
    den = pd.to_numeric(sw.get("followers"), errors="coerce")
    sw[f"er_{d}"] = np.where((den > 0) & num.notna(), num/den, np.nan)

# --- engagement_rate = базовый reposts/followers ---
sw["engagement_rate"] = np.where(
    (pd.to_numeric(sw.get("followers"), errors="coerce") > 0) &
    pd.to_numeric(sw.get("reposts"), errors="coerce").notna(),
    pd.to_numeric(sw.get("reposts"), errors="coerce") / pd.to_numeric(sw.get("followers"), errors="coerce"),
    np.nan
).astype(float)

# --- безопасная подстановка TG: reposts_2025-08-15 <- reposts, где NaN ---
if "reposts_2025-08-15" in sw.columns and "reposts" in sw.columns:
    m_fill = sw["reposts_2025-08-15"].isna() & sw["reposts"].notna()
    sw.loc[m_fill, "reposts_2025-08-15"] = pd.to_numeric(sw.loc[m_fill, "reposts"], errors="coerce")

# --- er_final: TG = reposts_2025-08-15 / followers; VK = reposts_2025-06-07 / followers ---
for c in ["followers","reposts_2025-08-15","reposts_2025-06-07"]:
    if c in sw.columns:
        sw[c] = pd.to_numeric(sw[c], errors="coerce")

sw["er_final"] = np.nan
m_tg = (sw["platform"] == "tg") & sw["reposts_2025-08-15"].notna() & (sw["followers"] > 0)
sw.loc[m_tg, "er_final"] = sw.loc[m_tg, "reposts_2025-08-15"] / sw.loc[m_tg, "followers"]

m_vk = (sw["platform"] == "vk") & sw["reposts_2025-06-07"].notna() & (sw["followers"] > 0)
sw.loc[m_vk, "er_final"] = sw.loc[m_vk, "reposts_2025-06-07"] / sw.loc[m_vk, "followers"]

sw["er_final"] = pd.to_numeric(sw["er_final"], errors="coerce")

# --- типы перед сохранением ---
for col in ["platform","source_name","source_link","published_at","text","link",
            "tg_channel_username","tg_fwd_src_name","image_url"]:
    if col in sw.columns:
        sw[col] = sw[col].astype("string")

for col in ["has_photo","has_video","has_link","has_poll","is_ad","tg_is_forwarded"]:
    if col in sw.columns:
        if col in sw.columns:
        # Для числовых значений: 0 -> False, 1 -> True
          if sw[col].dtype in [np.int64, np.float64]:
            sw[col] = sw[col].astype(bool)
        # Для строковых значений
          elif sw[col].dtype == 'object' or sw[col].dtype == 'string':
            sw[col] = sw[col].map(lambda x:
                True if pd.notna(x) and str(x).strip().lower() in ('1', 'true', 'yes', 't')
                else False if pd.notna(x) and str(x).strip().lower() in ('0', 'false', 'no', 'f')
                else pd.NA
            )
        # Для уже булевых значений
        else:
            sw[col] = sw[col].map(lambda x: bool(x) if pd.notna(x) else pd.NA)

num_like = [c for c in sw.columns if re.match(r"^(likes|comments|reposts|er)_\d{4}-\d{2}-\d{2}$", c)]
num_base = ["likes","comments","reposts","followers","vk_group_id","vk_from_id","tg_channel_id",
            "tg_fwd_src_id","tg_fwd_src_msg_id","text_len","engagement_rate","er_final"]
for col in num_base + num_like:
    if col in sw.columns:
        sw[col] = pd.to_numeric(sw[col], errors="coerce")

sw["post_id"] = pd.to_numeric(sw["post_id"], errors="coerce").astype("Int64")

# --- проверки целостности ---
# дубликаты TG по составному ключу
dup_tg = sw.loc[sw["platform"].eq("tg"), ["post_id","tg_channel_id","tg_channel_username"]].duplicated(keep=False).sum()
print("TG duplicates by (post_id, tg_channel_id, tg_channel_username):", int(dup_tg))

# доля NaN в ключевых снапшотах
keys = [c for c in ["reposts_2025-06-07","reposts_2025-08-15"] if c in sw.columns]
print("Non-null snapshot counts:", {k: int(sw[k].notna().sum()) for k in keys})

print("er_final non-null:", int(sw["er_final"].notna().sum()))
print("engagement_rate non-null:", int(sw["engagement_rate"].notna().sum()))

# --- сохранение (pandas сам экранирует кавычки, переносы строк не теряются) ---
sw.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
try:
    sw.to_parquet(OUT_PQ, index=False)
except Exception as e:
    print("Parquet save skipped:", e)

print("Saved:", OUT_CSV)
print("Saved:", OUT_PQ)
print("Shape:", sw.shape)


TG duplicates by (post_id, tg_channel_id, tg_channel_username): 0
Non-null snapshot counts: {'reposts_2025-06-07': 12457, 'reposts_2025-08-15': 21334}
er_final non-null: 21334
engagement_rate non-null: 21334
Saved: /content/social_wide_5.csv
Saved: /content/social_wide_5.parquet
Shape: (21361, 41)


In [ ]:
# --- проверка и коррекция булевых колонок после объединения ---
print("\n=== CHECKING BOOLEAN COLUMNS AFTER MERGE ===")

bool_cols = ["has_photo", "has_video", "has_link", "has_poll"]
for col in bool_cols:
    if col in sw.columns:
        print(f"\n{col} values before correction:")
        print(sw[col].value_counts(dropna=False).head())

        # Коррекция для VK данных (где были числа)
        vk_mask = (sw["platform"] == "vk") & (sw[col].notna())
        if vk_mask.any():
            # Преобразуем числовые значения в булевы
            sw.loc[vk_mask, col] = sw.loc[vk_mask, col].astype(str).str.strip().map({
                '0': False, '1': True, '0.0': False, '1.0': True,
                'false': False, 'true': True
            })

        # Коррекция для TG данных
        tg_mask = (sw["platform"] == "tg") & (sw[col].notna())
        if tg_mask.any():
            sw.loc[tg_mask, col] = sw.loc[tg_mask, col].astype(str).str.strip().map({
                'False': False, 'True': True, 'false': False, 'true': True,
                '0': False, '1': True
            })

        print(f"{col} values after correction:")
        print(sw[col].value_counts(dropna=False).head())

# --- проверка распределения по платформам ---
print("\n=== BOOLEAN COLUMNS BY PLATFORM ===")
for col in bool_cols:
    if col in sw.columns:
        print(f"\n{col}:")
        platform_stats = sw.groupby('platform')[col].value_counts(normalize=True)
        print(platform_stats)